# Account Update Analysis

This notebook analyzes the account update statistics generated by the `count_account_updates` tool.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Load the dataset
# Replace with your specific file if different
file_path = 'account_stats_1m.csv' 
try:
    df = pd.read_csv(file_path)
    print(f"Loaded {len(df):,} rows from {file_path}")
    print(df.head())
except FileNotFoundError:
    print(f"File {file_path} not found. Please ensure the CSV is in the output directory or update the path.")

## Basic Statistics

In [ ]:
if 'df' in locals():
    total_updates = df['UpdateCount'].sum()
    total_accounts = len(df)
    mean_updates = df['UpdateCount'].mean()
    median_updates = df['UpdateCount'].median()
    max_updates = df['UpdateCount'].max()

    print(f"Total Accounts: {total_accounts:,}")
    print(f"Total Updates: {total_updates:,}")
    print(f"Mean Updates per Account: {mean_updates:.2f}")
    print(f"Median Updates per Account: {median_updates:.2f}")
    print(f"Max Updates for a single Account: {max_updates:,}")
    
    # Percentiles
    print("\nPercentiles:")
    for p in [90, 95, 99, 99.9, 99.99]:
        val = np.percentile(df['UpdateCount'], p)
        print(f"{p}th percentile: {val:.0f}")

## Top 20 Most Updated Accounts

In [ ]:
if 'df' in locals():
    top_20 = df.nlargest(20, 'UpdateCount').copy()
    # Truncate addresses for better visualization
    top_20['ShortAddress'] = top_20['Address'].apply(lambda x: x[:6] + "..." + x[-4:] if isinstance(x, str) else str(x))

    plt.figure(figsize=(12, 8))
    sns.barplot(x='UpdateCount', y='ShortAddress', data=top_20, palette='viridis')
    plt.title('Top 20 Most Updated Accounts')
    plt.xlabel('Number of Updates')
    plt.ylabel('Account Address')
    plt.tight_layout()
    plt.show()

## Update Count Distribution

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(12, 6))
    # Log-Log scale works best for power-law distributions
    sns.histplot(df['UpdateCount'], bins=100, log_scale=True)
    plt.title('Distribution of Update Counts (Log-Log Scale)')
    plt.xlabel('Number of Updates (Log Scale)')
    plt.ylabel('Frequency (Log Scale)')
    plt.tight_layout()
    plt.show()

## Lorenz Curve & Gini Coefficient
Visualizing the inequality of updates (e.g., do a few accounts account for most updates?).

In [ ]:
if 'df' in locals():
    # Sort by update count
    sorted_df = df.sort_values('UpdateCount', ascending=True)

    # Cumulative sums
    cum_accounts = np.arange(1, len(sorted_df) + 1) / len(sorted_df)
    cum_updates = sorted_df['UpdateCount'].cumsum() / sorted_df['UpdateCount'].sum()
    
    # Gini Coefficient (NumPy 2.0+ trapz deprecated)
    if hasattr(np, 'trapezoid'):
        gini = 1 - 2 * np.trapezoid(cum_updates, cum_accounts)
    else:
        gini = 1 - 2 * np.trapz(cum_updates, cum_accounts)

    plt.figure(figsize=(8, 8))
    plt.plot(cum_accounts, cum_updates, label=f'Lorenz Curve (Gini: {gini:.4f})')
    plt.plot([0, 1], [0, 1], 'r--', label='Perfect Equality')
    plt.title(f'Lorenz Curve of Account Updates\nGini Coefficient: {gini:.4f}')
    plt.xlabel('Cumulative Fraction of Accounts')
    plt.ylabel('Cumulative Fraction of Updates')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    print(f"Gini Coefficient: {gini:.4f}")